In [ ]:
import jax

# Un/comment this for double/single precision:
# jax.config.update('jax_enable_x64', True)

import copy
import equinox as eqx
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import UncertainSCI._equinox as _eqx
import UncertainSCI.gp as gp


D = 1  
C = 1

np.random.seed(0x01234567)
seed = 0xdeadbeef  # Used in gp.GaussianProcess initialization.

FIGSIZE = (7, 7 / 1.6)
FIGDPI = None

def print_loss_here(g):  # Simple helper for this notebook; does not generalize.
    print(
        f'Loss at D = {jnp.squeeze(g.k.D()):.4e} (length scale {1 / jnp.squeeze(g.k.D())**2:.4e}): '
        f'{g.loss(x_train, y_train, s_train):.4e}'
    )


In [ ]:
x_plot = jnp.linspace(0, 2 * jnp.pi, 1000).reshape((-1, D))

def f_hidden(x):
    return 2 * jnp.cos(x) * jnp.sin(4 * x)

def f_noisy(x):
    y = f_hidden(x)
    return (
        y,
        1e-3 * jnp.ones_like(y)
    )


In [ ]:
mu = gp.mean.Affine(
    dim=D,
    cdim=C,
    a=0. * jnp.ones((C, D)),
    b=0.,
    a_is_static=True,
    b_is_static=True
)
k = gp.kernel.Gaussian(
    dim=D,
    cdim=C,
    D=jnp.ones((1, 1)),
)
g = gp.GaussianProcess(dim=D, cdim=C, mu=mu, k=k, seed=seed, nugget=1e-3)


In [ ]:
def gp_stats(g: gp.GaussianProcess, x, which='posterior', p=100):
    if which == 'posterior':
        m = jnp.squeeze(g.posterior_mean(x))
        c = jnp.squeeze(jnp.diag(g.posterior_covariance(x, x)))
        r = jnp.squeeze(g.posterior_realization(x, p))
    elif which == 'prior':
        m = jnp.squeeze(g.prior_mean(x))
        c = jnp.squeeze(jnp.diag(g.prior_covariance(x, x)))
        r = jnp.squeeze(g.prior_realization(x, p))
    else:
        raise ValueError

    return m, c, r


def plot_distribution_mean(ax, g: gp.GaussianProcess, x, which='posterior', p=100):
    m, c, r = gp_stats(g, x, which, p)
    ax.fill_between(
        x.squeeze(),
        (m - c),
        (m + c),
        color='#00000040',
        edgecolor='none'
    )
    ax.plot(
        x.squeeze(),
        r,
        color='#00000010'
    )
    ax.plot(
        x.squeeze(),
        f_hidden(x).squeeze(),
    )
    if which == 'posterior':
        ax.errorbar(
            g.x_train.squeeze(),
            g.y_train.squeeze(),
            yerr=g.s_train.squeeze(),
            capsize=2,
            linestyle='none'
        )
    ax.set_title('Distribution')


def plot_distribution_variance(ax, g: gp.GaussianProcess, x, which='posterior', p=100, colorlast=True):
    m, c, r = gp_stats(g, x, which, p)
    ax.plot(
        x.squeeze(),
        c
    )
    if which == 'posterior':
        if colorlast:
            ax.vlines(g.x_train.squeeze()[:-1], 0, 1, color='tab:green')
            ax.vlines(g.x_train.squeeze()[-1], 0, 1, color='tab:red')
        else:
            ax.vlines(g.x_train.squeeze(), 0, 1, color='tab:green')
    ax.set_title('Variance')


def plot_loss_landscape(
        ax,
        g: gp.GaussianProcess,
        p_name: tuple[str, ...],
        p_range: jax.Array,
        x: jax.Array | None = None,
        ymin: jax.Array | None = None,
        ymax: jax.Array | None = None,
        ytop: float = 1e2
    ):
    """
    Plot the loss landscape with respect to the parameter in Gaussian process ``g``
    at location ``p_name``.

    Performs recursive ``getattr`` until ``p_name`` is exhausted.

    Args:
        ax (matplotlib.axes.Axes):
            The axes to plot into.
        g (gp.GaussianProcess):
            The Gaussian process to evaluate.
        p_name (tuple of str):
            Attr name chain leading to the parameter (:class:`jax.Array` or :class:`_eqx.ComputedArray`)
            to be sampled.
        p_range (array):
            Range of parameter values to test.
        x (array or float, optional):
            x values at which to place vlines, default ``None``.
        ymin (array or float, optional):
            y range for vlines, must match shape of x, default ``None``.
        ymax (array or float, optional):
            y range for vlines, must match shape of x, default ``None``.
        ytop (float, optional):
            Top of y-axis to plot, default 1e2.
    """
    p_parent = None
    g = copy.deepcopy(g)
    p = g
    for name in p_name:
        p_parent = p
        p = getattr(p, name)

    if p_parent is None:
        raise ValueError(
            f'Did not find parent of target from keys {p_name}! '
            f'This usually happens when keys {p_name} has zero length.'
        )

    if isinstance(p, _eqx.ComputedArray):
        numel = p().size
        shape = p().shape
    elif isinstance(p, jax.Array):
        numel = p.size
        shape = p.shape
    else:
        raise ValueError(
            f'Resolution of target from keys {p_name} led to unsupported type {type(p)}!'
        )

    if numel > 1:
        raise NotImplementedError('Loss landscape for non-scalar parameter is not supported!')

    losses = jnp.empty(p_range.shape)
    for i, p_value in enumerate(p_range):
        if isinstance(p, _eqx.ComputedArray):
            p_value = type(p).from_array(p_value * jnp.ones(shape))
        else:
            p_value = p_value * jnp.ones(shape)

        setattr(p_parent, p_name[-1], p_value)
        losses = losses.at[i].set(g.loss(g.x_train, g.y_train, g.s_train))

    ax.plot(p_range, losses)
    if x is not None:
        if ymin is None:
            ymin = min(losses)
        if ymax is None:
            ymax = max(losses)
        ax.vlines(x, ymin, ymax)
    ax.set_xscale('symlog')
    ax.set_yscale('symlog')
    ax.set_ylim(top=ytop)
    ax.set_title('Loss Landscape')
    ax.set_xlabel(p_name[-1])


def plot_in_subplots(nrows=1, ncols=1, **kwargs):
    """
    Returns ``(fig, axes)`` with plot scaled correctly for ``(nrows, ncols)``.
    """
    if 'figsize' in kwargs:
        raise ValueError("kwargs had 'figsize' key: that's the whole point of this function!")
    return plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * FIGSIZE[0], nrows * FIGSIZE[1]),
        **kwargs
    )


In [ ]:
fig, (ax1, ax2) = plot_in_subplots(2, 1, dpi=FIGDPI)
plot_distribution_mean(ax1, g, x_plot, 'prior')
plot_distribution_variance(ax2, g, x_plot, 'prior')
plt.show()


In [ ]:
N_INIT = 10

x_train = jnp.linspace(0, 2 * jnp.pi, N_INIT).reshape(-1, D)
y_train, s_train = f_noisy(x_train)    

g.condition(x_train, y_train, s_train)


Plot what that looks like:

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
plot_distribution_mean(ax1, g, x_plot, 'prior')
plot_distribution_variance(ax2, g, x_plot, 'prior')
plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


Tune:

In [ ]:
losses = g.tune()

plt.figure(figsize=FIGSIZE)
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()


Plot what things look like after tuning:

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
plot_distribution_mean(ax1, g, x_plot, 'posterior')
plot_distribution_variance(ax2, g, x_plot, 'posterior', colorlast=False)
plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


Iteratively sample points to choose for tuning, then tune:

In [ ]:
N_TRAIN = 10
N_SAMPLE = 20

x_sample = jnp.linspace(0, 2 * jnp.pi, N_SAMPLE).reshape(-1, D)

# def covariance(...): ...

# def optimize_x_over_covariance(...): ...

for i in range(N_TRAIN):

    # ...do those things here...

    _, c, _ = gp_stats(g, x_sample, p=0)  # Greedy sampling on dense mesh.
    _x = x_sample[jnp.argmax(c)].reshape(-1, D)
    _y, _s = f_noisy(_x)

    x_train = jnp.concat((x_train, _x))
    y_train = jnp.concat((y_train, _y))
    s_train = jnp.concat((s_train, _s))

    g.condition(x_train, y_train, s_train)
    g.tune()

    fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
    plot_distribution_mean(ax1, g, x_plot, 'posterior')
    plot_distribution_variance(ax2, g, x_plot, 'posterior')
    plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
    plt.show()
    print_loss_here(g)
    print('\n' * 3)
